# Answer / Critique / Refine Workflow (Challenge 4)

A question-answering pipeline built from ADK **workflow agents**: a
`SequentialAgent` runs a `search_agent`, a `critique_agent`, and a
`refine_agent` in a fixed order, and a `greeter` (root `LlmAgent`) is the
entry point that forwards every question straight to that pipeline.

1. **`search_agent`** — uses Google Search to draft an initial answer.
2. **`critique_agent`** — reviews that answer and lists concrete
   improvements, without rewriting it.
3. **`refine_agent`** — rewrites the answer, applying the critique.

Each step's output is stored in session state (`initial_answer`,
`critique`, `final_answer` via each agent's `output_key`) so the next step
can reference it directly in its instruction — this is what "verifies and
refines the answer before returning it" means here: the user only ever
sees `final_answer`.

> Run this notebook in **Colab Enterprise** inside your Cloud Skills Boost
> project, same as the previous challenges.


## 1. Install dependencies

In [ ]:
%pip install --quiet google-adk


## 2. Configuration

This notebook only calls Gemini (no weather tools, no Maps key, no
third-party model), so the only setup needed is Vertex AI auth — this
notebook's existing GCP credentials via Colab Enterprise, no personal key
required.


In [ ]:
import os

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

if "GOOGLE_CLOUD_PROJECT" not in os.environ:
    import subprocess

    try:
        _detected_project = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if _detected_project and _detected_project != "(unset)":
            os.environ["GOOGLE_CLOUD_PROJECT"] = _detected_project
    except (subprocess.CalledProcessError, FileNotFoundError):
        pass

print("Vertex project:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("Vertex location:", os.environ.get("GOOGLE_CLOUD_LOCATION"))


## 3. Resolve an available Gemini model

Same probing approach as the previous challenges: try a shortlist of
Gemini model/region combinations on Vertex AI and use the first one this
project actually has access to.


In [ ]:
from google import genai
from google.genai.errors import ClientError

_GEMINI_MODEL_CANDIDATES = [
    "gemini-2.5-flash-lite",  # confirmed available in Model Garden
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.5-flash-lite",
    "gemini-2.0-flash-001",
    "gemini-2.0-flash",
    "gemini-1.5-flash-002",
]
_GEMINI_REGION_CANDIDATES = ["us-central1", "us-east4", "us-east5", "us-west1", "europe-west4"]

_project = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL_ID = None
GEMINI_LOCATION = None

for _region in _GEMINI_REGION_CANDIDATES:
    _candidate_client = genai.Client(vertexai=True, project=_project, location=_region)
    for _model in _GEMINI_MODEL_CANDIDATES:
        try:
            _candidate_client.models.generate_content(model=_model, contents="ping")
        except ClientError as exc:
            if exc.code == 404:
                print(f"unavailable: {_model} in {_region} (404)")
                continue
            raise
        MODEL_ID = _model
        GEMINI_LOCATION = _region
        break
    if MODEL_ID:
        break

if MODEL_ID is None:
    raise RuntimeError(
        "No Gemini model/region combination on Vertex AI worked for "
        f"project {_project}. Check Vertex AI > Model Garden in the "
        "console for what's actually enabled and add it to "
        "_GEMINI_MODEL_CANDIDATES/_GEMINI_REGION_CANDIDATES above."
    )

os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
print(f"Using Gemini model: {MODEL_ID} in {GEMINI_LOCATION}")


## 4. Search agent

Drafts an initial answer using ADK's built-in Google Search tool.
`output_key="initial_answer"` tells ADK to save this agent's final
response text into `session.state["initial_answer"]` automatically, so the
next agent in the pipeline can reference it.


In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search

search_agent = Agent(
    name="search_agent",
    model=MODEL_ID,
    description="Finds up-to-date information via Google Search to draft an initial answer.",
    instruction="""Use the google_search tool to find current, accurate
information that answers the user's question. Write a clear, well
organized initial answer based on what you find. This is a first draft —
it will be reviewed and improved by other agents next, so focus on
getting the facts right rather than polishing the wording.""",
    tools=[google_search],
    output_key="initial_answer",
)


## 5. Critique agent

Reviews the initial answer and lists concrete improvements — it does not
rewrite anything itself. Its instruction references `{initial_answer}`,
which ADK substitutes from session state at runtime. Its own output is
saved to `session.state["critique"]` via `output_key`.


In [ ]:
critique_agent = Agent(
    name="critique_agent",
    model=MODEL_ID,
    description="Reviews the initial answer and suggests concrete improvements.",
    instruction="""You are a critical reviewer. Read the initial answer
below and identify concrete ways it could be improved: missing
information, unclear wording, unsupported claims, or organization
problems. If it's already solid, say so briefly. Do not rewrite the
answer yourself — only list specific, actionable suggestions.

Initial answer:
\"\"\"
{initial_answer}
\"\"\"
""",
    output_key="critique",
)


## 6. Refine agent

Rewrites the initial answer, applying the critique's suggestions, and
produces the final answer the user actually sees. Its own output is saved
to `session.state["final_answer"]`.


In [ ]:
refine_agent = Agent(
    name="refine_agent",
    model=MODEL_ID,
    description="Rewrites the initial answer using the critique's suggestions.",
    instruction="""Rewrite the answer below, applying every applicable
suggestion from the critique. Produce a single polished, final answer for
the user — do not mention the review process, the critique, or any other
agent in your output; just answer the original question well.

Initial answer:
\"\"\"
{initial_answer}
\"\"\"

Critique / suggested improvements:
\"\"\"
{critique}
\"\"\"
""",
    output_key="final_answer",
)


## 7. Answer team (SequentialAgent) and greeter (root agent)

`SequentialAgent` runs `search_agent`, `critique_agent`, then
`refine_agent` in that fixed order — no LLM decides the order, so there's
no risk of the transfer/tool conflict a dynamically-routing root agent
would have with `google_search` (that's the issue Challenge 3 ran into:
here, `search_agent`'s direct parent is a workflow agent, not an `LlmAgent`
that needs its own transfer tool). `greeter` is the entry point a user
actually talks to; it has no tools of its own and simply hands every
question to the answer team as a sub agent.


In [ ]:
from google.adk.agents import SequentialAgent

answer_team = SequentialAgent(
    name="answer_team",
    description="Answers a question, critiques the initial answer, then refines it.",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

greeter = Agent(
    name="greeter",
    model=MODEL_ID,
    description="Entry point that forwards user questions to the answer team.",
    instruction="""You are the entry point for a question-answering
assistant. For every question the user asks, delegate immediately to the
answer team rather than answering yourself.""",
    sub_agents=[answer_team],
)


## 8. Test harness

Runs a question through `greeter` and prints the raw event stream — each
event's `author` shows which agent (`greeter`, `search_agent`,
`critique_agent`, or `refine_agent`) produced it, so you can see the whole
answer -> critique -> refine pipeline execute step by step, not just the
final text. It then also prints the session's final state, showing all
three stored outputs side by side.


In [ ]:
import asyncio

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

APP_NAME = "answer_workflow_app"
USER_ID = "test_user"

session_service = InMemorySessionService()


async def run_and_print_events(agent: Agent, query: str, session_id: str) -> None:
    """Run one query through an ADK agent and print every event it emits.

    Args:
        agent: The (root) agent to run the query through.
        query: The user message to send.
        session_id: A unique session id for this run.
    """
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=query)])

    print(f"\n--- Query: {query} ---")
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        author = getattr(event, "author", "?")
        parts = event.content.parts if event.content else []
        for part in parts:
            if getattr(part, "function_call", None):
                fc = part.function_call
                print(f"[{author}] FUNCTION_CALL: {fc.name}({dict(fc.args or {})})")
            if getattr(part, "function_response", None):
                fr = part.function_response
                print(f"[{author}] FUNCTION_RESPONSE: {fr.name} -> (truncated)")
            if getattr(part, "text", None):
                tag = "FINAL" if event.is_final_response() else "TEXT"
                print(f"[{author}] {tag}: {part.text.strip()[:400]}")

    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    print("\n--- Session state after the run ---")
    for key in ("initial_answer", "critique", "final_answer"):
        print(f"\n[{key}]\n{session.state.get(key)}")


TEST_QUERY = "What are the main new capabilities in the latest Gemini model family?"


async def run_tests() -> None:
    await run_and_print_events(greeter, TEST_QUERY, session_id="greeter-session-0")


await run_tests()
